In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset
from torchvision import datasets, transforms, models
import os
from PIL import Image
from tqdm import tqdm

class ImageFolderWithTxt(Dataset):
    def __init__(self, root_dir, txt_path, transform=None):
        """
        root_dir: 图片文件夹路径
        txt_path: 标签文件路径
        transform: 图像预处理函数（比如 ToTensor(), Resize(), Normalize() 等）
        """
        self.root_dir = root_dir
        self.transform = transform

        # 读取 txt 文件，存成 [(img_path, label), ...]
        self.samples = []
        with open(txt_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                img_name, label = line.split(',')
                self.samples.append((img_name, label))
        all_labels = sorted(set(label for _, label in self.samples))
        self.label_to_id = {label: idx for idx, label in enumerate(all_labels)}
        self.id_to_label = {idx: label for label, idx in self.label_to_id.items()}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.root_dir, img_name)

        # 打开图像
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        label_id = self.label_to_id[label]
        return {
            "image": image,
            "label": torch.tensor(label_id, dtype=torch.long),
            "filename": img_name
        }


data_path = "classify-leaves"

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomVerticalFlip(),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
])


val_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
])

if __name__ == "__main__":
    dataset = ImageFolderWithTxt(
        root_dir="classify-leaves",
        txt_path="classify-leaves/train.csv",
        transform=train_transform
    )
    # print(dataset.__len__())
    # item = dataset[0]
    # print("数字标签:", item["label"].item())
    # print("字符串标签:", dataset.id_to_label[item["label"].item()])
    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=True,
        num_workers=4,  # 或先改成0测试
    )

    for batch in loader:
        imgs = batch["image"]
        labels = batch["label"]
        names = batch["filename"]
        print(names[0], labels[0])

